# Step 2 — Connect to SQL Server (`mvkhc`) via Spark JDBC

Second step of the Synthea CSV -> SQL Server pipeline.
Reads connection settings from `hc_bigdata/config/sql_connection_config.json` and opens a Spark JDBC connection to the local `SQLEXPRESS` instance, database `mvkhc`, using Windows Integrated Security. This only tests the connection — no data written yet.

In [1]:
import json

with open("../../../config/sql_connection_config.json") as f:
    config = json.load(f)["sql_server"]

config

{'host': 'localhost',
 'instance': 'SQLEXPRESS',
 'database': 'mvkhc',
 'auth_mode': 'windows_integrated',
 'driver_class': 'com.microsoft.sqlserver.jdbc.SQLServerDriver',
 'jdbc_url': 'jdbc:sqlserver://localhost\\SQLEXPRESS;databaseName=mvkhc;integratedSecurity=true;encrypt=true;trustServerCertificate=true;',
 'maven_package': 'com.microsoft.sqlserver:mssql-jdbc:12.6.1.jre11'}

In [5]:
%pip install pyspark.sql

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pyspark.sql (from versions: none)
ERROR: No matching distribution found for pyspark.sql

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("synthea_sql_connection") \
    .master("local[*]") \
    .config("spark.jars.packages", config["maven_package"]) \
    .getOrCreate()

spark

In [ ]:
df = spark.read.jdbc(
    url=config["jdbc_url"],
    table="(SELECT 1 AS test_col) AS connection_check",
    properties={"driver": config["driver_class"]},
)
df.show()
print(df)

In [ ]:
spark.stop()